In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, avg, max as spark_max, min as spark_min, round as spark_round, date_format

def main():
    # Create Spark session with local master and file system configuration
    spark = (
        SparkSession.builder
        .appName("read_gold_fact_orders")
        .master("local[*]")
        .config("spark.hadoop.fs.defaultFS", "file:///")
        .getOrCreate()
    )
    
    # Define the path to the gold fact table
    gold_path = "file:/data/gold/fact_orders"
    
    # Print header information
    print("Reading Gold fact table: fact_orders")
    print(f"Path: {gold_path}")
    
    # Read the parquet files from gold layer
    df = spark.read.parquet(gold_path)
    
    # Display the schema of the fact table
    print("\nSchema:")
    df.printSchema()
    
    # Show sample data from all records
    print("\nSample data (first 20 records):")
    df.show(20, truncate=False)
    
    # Display fact table statistics
    print("\nFact table statistics:")
    df.describe().show()
    
    # Count total records in fact table
    total_records = df.count()
    print(f"\nTotal records in fact table: {total_records}")
    
    # Create a temporary view for SQL queries
    print("\nCreating temp view: fact_orders")
    df.createOrReplaceTempView("fact_orders")
    
    # Read dimension tables for enriched queries
    print("\nLoading dimension tables...")
    
    # Read dim_customers
    dim_customers_path = "file:/data/gold/dim_customers"
    dim_customers = spark.read.parquet(dim_customers_path)
    dim_customers.createOrReplaceTempView("dim_customers")
    print("dim_customers loaded")
    
    # Read dim_products
    dim_products_path = "file:/data/gold/dim_products"
    dim_products = spark.read.parquet(dim_products_path)
    dim_products.createOrReplaceTempView("dim_products")
    print("dim_products loaded")
    
    # Query: Total sales by customer (name), product (description) and date (dd-mm-yyyy)
    print("\n" + "="*80)
    print("SALES BY CUSTOMER NAME, PRODUCT DESCRIPTION AND DATE")
    print("="*80)
    spark.sql("""
        SELECT
            c.customer_name,
            p.product_description,
            DATE_FORMAT(f.order_date, 'dd-MM-yyyy') as order_date_formatted,
            SUM(f.total_quantity) as total_quantity,
            ROUND(SUM(f.total_sales), 2) as total_sales
        FROM fact_orders f
        INNER JOIN dim_customers c 
            ON f.customer_code = c.customer_code 
            AND c.is_current = true
        INNER JOIN dim_products p 
            ON f.product_code = p.product_code 
            AND p.is_current = true
        GROUP BY c.customer_name, p.product_description, DATE_FORMAT(f.order_date, 'dd-MM-yyyy')
        ORDER BY DATE_FORMAT(f.order_date, 'dd-MM-yyyy') DESC, total_sales DESC
        LIMIT 50
    """).show(truncate=False)
    
    # Query: Total sales summary by date with customer and product names
    print("\n" + "="*80)
    print("DAILY SALES DETAIL WITH NAMES (LAST 10 DAYS)")
    print("="*80)
    spark.sql("""
        SELECT
            DATE_FORMAT(f.order_date, 'dd-MM-yyyy') as sale_date,
            c.customer_name,
            p.product_description,
            f.total_quantity,
            ROUND(f.total_sales, 2) as total_sales
        FROM fact_orders f
        INNER JOIN dim_customers c 
            ON f.customer_code = c.customer_code 
            AND c.is_current = true
        INNER JOIN dim_products p 
            ON f.product_code = p.product_code 
            AND p.is_current = true
        ORDER BY f.order_date DESC, total_sales DESC
        LIMIT 100
    """).show(truncate=False)
    
    # Query: Aggregated sales by customer name
    print("\n" + "="*80)
    print("SALES TOTALS BY CUSTOMER NAME")
    print("="*80)
    spark.sql("""
        SELECT
            c.customer_name,
            COUNT(DISTINCT f.order_code) as total_orders,
            SUM(f.total_quantity) as total_items,
            ROUND(SUM(f.total_sales), 2) as total_revenue
        FROM fact_orders f
        INNER JOIN dim_customers c 
            ON f.customer_code = c.customer_code 
            AND c.is_current = true
        GROUP BY c.customer_name
        ORDER BY total_revenue DESC
        LIMIT 20
    """).show(truncate=False)
    
    # Query: Aggregated sales by product description
    print("\n" + "="*80)
    print("SALES TOTALS BY PRODUCT DESCRIPTION")
    print("="*80)
    spark.sql("""
        SELECT
            p.product_description,
            COUNT(DISTINCT f.order_code) as times_ordered,
            SUM(f.total_quantity) as total_quantity_sold,
            ROUND(SUM(f.total_sales), 2) as total_revenue
        FROM fact_orders f
        INNER JOIN dim_products p 
            ON f.product_code = p.product_code 
            AND p.is_current = true
        GROUP BY p.product_description
        ORDER BY total_revenue DESC
        LIMIT 20
    """).show(truncate=False)
    
    # Query: Sales by date with aggregations
    print("\n" + "="*80)
    print("SALES TOTALS BY DATE")
    print("="*80)
    spark.sql("""
        SELECT
            DATE_FORMAT(f.order_date, 'dd-MM-yyyy') as sale_date,
            COUNT(DISTINCT f.order_code) as total_orders,
            COUNT(DISTINCT f.customer_code) as unique_customers,
            COUNT(DISTINCT f.product_code) as unique_products,
            SUM(f.total_quantity) as total_items,
            ROUND(SUM(f.total_sales), 2) as daily_revenue
        FROM fact_orders f
        GROUP BY DATE_FORMAT(f.order_date, 'dd-MM-yyyy')
        ORDER BY f.order_date DESC
        LIMIT 30
    """).show(truncate=False)
    
    # Query: OVERALL SALES SUMMARY
    print("\n" + "="*80)
    print("OVERALL SALES SUMMARY")
    print("="*80)
    spark.sql("""
        SELECT
            COUNT(DISTINCT order_code) as total_orders,
            COUNT(DISTINCT customer_code) as unique_customers,
            COUNT(DISTINCT product_code) as unique_products,
            SUM(total_quantity) as total_items_sold,
            ROUND(SUM(total_sales), 2) as total_revenue,
            ROUND(AVG(total_sales), 2) as avg_order_value,
            ROUND(MAX(total_sales), 2) as max_order_value,
            ROUND(MIN(total_sales), 2) as min_order_value
        FROM fact_orders
    """).show(truncate=False)
    
    # Query: Top 10 customers by revenue
    print("\n" + "="*80)
    print("TOP 10 CUSTOMERS BY TOTAL REVENUE")
    print("="*80)
    spark.sql("""
        SELECT
            customer_code,
            COUNT(DISTINCT order_code) as total_orders,
            SUM(total_quantity) as total_items,
            ROUND(SUM(total_sales), 2) as total_revenue,
            ROUND(AVG(total_sales), 2) as avg_order_value
        FROM fact_orders
        GROUP BY customer_code
        ORDER BY total_revenue DESC
        LIMIT 10
    """).show(truncate=False)
    
    # Query: Top 10 products by revenue
    print("\n" + "="*80)
    print("TOP 10 PRODUCTS BY TOTAL REVENUE")
    print("="*80)
    spark.sql("""
        SELECT
            product_code,
            COUNT(DISTINCT order_code) as times_ordered,
            SUM(total_quantity) as total_quantity_sold,
            ROUND(SUM(total_sales), 2) as total_revenue,
            ROUND(AVG(total_sales), 2) as avg_sale_value
        FROM fact_orders
        GROUP BY product_code
        ORDER BY total_revenue DESC
        LIMIT 10
    """).show(truncate=False)
    
    # Query: Top 10 products by quantity sold
    print("\n" + "="*80)
    print("TOP 10 PRODUCTS BY QUANTITY SOLD")
    print("="*80)
    spark.sql("""
        SELECT
            product_code,
            SUM(total_quantity) as total_quantity_sold,
            COUNT(DISTINCT order_code) as times_ordered,
            ROUND(SUM(total_sales), 2) as total_revenue
        FROM fact_orders
        GROUP BY product_code
        ORDER BY total_quantity_sold DESC
        LIMIT 10
    """).show(truncate=False)
    
    # Query: Sales by date (last 10 days)
    print("\n" + "="*80)
    print("DAILY SALES SUMMARY (LAST 10 DAYS)")
    print("="*80)
    spark.sql("""
        SELECT
            DATE(order_date) as sales_date,
            COUNT(DISTINCT order_code) as total_orders,
            COUNT(DISTINCT customer_code) as unique_customers,
            SUM(total_quantity) as total_items,
            ROUND(SUM(total_sales), 2) as daily_revenue,
            ROUND(AVG(total_sales), 2) as avg_order_value
        FROM fact_orders
        GROUP BY DATE(order_date)
        ORDER BY sales_date DESC
        LIMIT 10
    """).show(truncate=False)
    
    # Query: Orders with highest value
    print("\n" + "="*80)
    print("TOP 10 HIGHEST VALUE ORDERS")
    print("="*80)
    spark.sql("""
        SELECT
            order_code,
            customer_code,
            order_date,
            COUNT(DISTINCT product_code) as products_in_order,
            SUM(total_quantity) as total_items,
            ROUND(SUM(total_sales), 2) as order_total
        FROM fact_orders
        GROUP BY order_code, customer_code, order_date
        ORDER BY order_total DESC
        LIMIT 10
    """).show(truncate=False)
    
    # Query: Customer purchase frequency
    print("\n" + "="*80)
    print("CUSTOMER PURCHASE FREQUENCY DISTRIBUTION")
    print("="*80)
    spark.sql("""
        SELECT
            order_count,
            COUNT(*) as number_of_customers
        FROM (
            SELECT
                customer_code,
                COUNT(DISTINCT order_code) as order_count
            FROM fact_orders
            GROUP BY customer_code
        )
        GROUP BY order_count
        ORDER BY order_count DESC
    """).show(truncate=False)
    
    # Query: Product diversity per order
    print("\n" + "="*80)
    print("PRODUCTS PER ORDER DISTRIBUTION")
    print("="*80)
    spark.sql("""
        SELECT
            products_per_order,
            COUNT(*) as number_of_orders,
            ROUND(AVG(order_total), 2) as avg_order_value
        FROM (
            SELECT
                order_code,
                COUNT(DISTINCT product_code) as products_per_order,
                SUM(total_sales) as order_total
            FROM fact_orders
            GROUP BY order_code
        )
        GROUP BY products_per_order
        ORDER BY products_per_order
    """).show(truncate=False)
    
    # Query: Monthly sales trend (if data spans multiple months)
    print("\n" + "="*80)
    print("MONTHLY SALES TREND")
    print("="*80)
    spark.sql("""
        SELECT
            YEAR(order_date) as year,
            MONTH(order_date) as month,
            COUNT(DISTINCT order_code) as total_orders,
            COUNT(DISTINCT customer_code) as unique_customers,
            SUM(total_quantity) as total_items,
            ROUND(SUM(total_sales), 2) as monthly_revenue
        FROM fact_orders
        GROUP BY YEAR(order_date), MONTH(order_date)
        ORDER BY year DESC, month DESC
    """).show(truncate=False)
    
    # Query: Example of a specific order detail
    print("\n" + "="*80)
    print("EXAMPLE: DETAILED VIEW OF ORDER #1")
    print("="*80)
    spark.sql("""
        SELECT
            order_code,
            customer_code,
            order_date,
            product_code,
            total_quantity,
            total_sales,
            line_count
        FROM fact_orders
        WHERE order_code = 1
        ORDER BY product_code
    """).show(truncate=False)
    
    # Stop the Spark session
    spark.stop()

if __name__ == "__main__":
    main()

Reading Gold fact table: fact_orders
Path: file:/data/gold/fact_orders

Schema:
root
 |-- order_code: integer (nullable = true)
 |-- customer_code: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- product_code: integer (nullable = true)
 |-- total_quantity: long (nullable = true)
 |-- total_sales: decimal(38,2) (nullable = true)
 |-- line_count: long (nullable = true)


Sample data (first 20 records):
+----------+-------------+-------------------+------------+--------------+-----------+----------+
|order_code|customer_code|order_date         |product_code|total_quantity|total_sales|line_count|
+----------+-------------+-------------------+------------+--------------+-----------+----------+
|2521      |123          |2024-01-15 10:00:00|24          |4             |1472.00    |1         |
|2834      |130          |2022-02-15 10:00:00|38          |3             |1368.00    |1         |
|2982      |133          |2022-06-15 10:00:00|34          |4             |3712

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `f`.`order_date` cannot be resolved. Did you mean one of the following? [`sale_date`, `total_items`, `total_orders`, `daily_revenue`, `unique_products`].; line 11 pos 17;
'GlobalLimit 30
+- 'LocalLimit 30
   +- 'Sort ['f.order_date DESC NULLS LAST], true
      +- Aggregate [date_format(order_date#1397, dd-MM-yyyy, Some(Etc/UTC))], [date_format(order_date#1397, dd-MM-yyyy, Some(Etc/UTC)) AS sale_date#2313, count(distinct order_code#1395) AS total_orders#2314L, count(distinct customer_code#1396) AS unique_customers#2315L, count(distinct product_code#1398) AS unique_products#2316L, sum(total_quantity#1399L) AS total_items#2317L, round(sum(total_sales#1400), 2) AS daily_revenue#2318]
         +- SubqueryAlias f
            +- SubqueryAlias fact_orders
               +- View (`fact_orders`, [order_code#1395,customer_code#1396,order_date#1397,product_code#1398,total_quantity#1399L,total_sales#1400